In [1]:

import torch
import platform

print("Python Version:", platform.python_version())
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("Please enable GPU in Runtime settings.")


Python Version: 3.13.16
PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Name: Tesla T4
GPU Memory: 14.56 GB


In [2]:

!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q openai-whisper


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:

!pip install -q unsloth


In [4]:

import torch
import whisper
import unsloth
import subprocess

print("Whisper:", whisper.__version__ if hasattr(whisper, "__version__") else "Imported")
print("Unsloth:", unsloth.__version__)
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("\nFFmpeg:")
subprocess.run(["ffmpeg", "-version"], capture_output=False)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1110 05:56:16.412000 1711 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 05:56:16.452000 1711 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Whisper: 20250625
Unsloth: 2026.11.3
PyTorch: 2.14.1+cu130
CUDA Available: True
GPU: Tesla T4

FFmpeg:


CompletedProcess(args=['ffmpeg', '-version'], returncode=0)

In [5]:

from google.colab import files
from IPython.display import Audio, display

uploaded = files.upload()

audio_path = next(iter(uploaded.keys()))

print("Uploaded audio:", audio_path)
print("Audio upload successful!")

display(Audio(filename=audio_path))


Saving sample_query.wav to sample_query.wav
Uploaded audio: sample_query.wav
Audio upload successful!


In [6]:

import whisper
import torch

# Load Whisper Base model
whisper_model = whisper.load_model("base")

print("Whisper Base model loaded successfully!")

# Transcribe uploaded audio
result = whisper_model.transcribe(
    audio_path,
    fp16=torch.cuda.is_available()
)

transcribed_text = result["text"].strip()

print("\n========== WHISPER TRANSCRIPTION ==========")
print(transcribed_text)
print("===========================================")


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 326MiB/s]


Whisper Base model loaded successfully!

========== WHISPER TRANSCRIPTION ==========
If a person has five apples and gives two apples to a friend, how many apples are left?


In [7]:

import gc
import torch

# Move Whisper to CPU to free GPU memory
whisper_model.to("cpu")

# Clear unused GPU memory
gc.collect()
torch.cuda.empty_cache()

print("Whisper moved to CPU successfully.")
print("GPU memory cleared.")

if torch.cuda.is_available():
    print(
        "GPU Memory Allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )
    print(
        "GPU Memory Reserved:",
        round(torch.cuda.memory_reserved() / 1024**3, 2),
        "GB"
    )


Whisper moved to CPU successfully.
GPU memory cleared.
GPU Memory Allocated: 0.01 GB
GPU Memory Reserved: 0.02 GB


In [8]:

# Optimize model for inference
FastLanguageModel.for_inference(model)

# Detect 4-bit quantized linear layers
linear4bit_count = sum(
    1 for module in model.modules()
    if module.__class__.__name__ == "Linear4bit"
)

print("Inference mode enabled.")
print("Detected Linear4bit modules:", linear4bit_count)

print("\nGPU Memory Usage:")
print("Allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("Reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")


NameError: name 'FastLanguageModel' is not defined

In [9]:

from unsloth import FastLanguageModel
import torch

print("FastLanguageModel imported successfully!")


FastLanguageModel imported successfully!


In [10]:

print("Model exists:", "model" in globals())
print("Tokenizer exists:", "tokenizer" in globals())


Model exists: False
Tokenizer exists: False


In [11]:

FastLanguageModel.for_inference(model)

linear4bit_count = sum(
    1 for module in model.modules()
    if module.__class__.__name__ == "Linear4bit"
)

print("Inference mode enabled!")
print("Detected Linear4bit modules:", linear4bit_count)

print("GPU Allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("GPU Reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")


NameError: name 'model' is not defined

In [12]:

import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


CUDA: True
GPU: Tesla T4


In [13]:

import torch
import gc
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=torch.float16,
    load_in_4bit=True,
)

print("Qwen model loaded successfully!")
print("Model:", model_name)
print("Quantization: 4-bit")


==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen model loaded successfully!
Model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
Quantization: 4-bit


In [14]:

FastLanguageModel.for_inference(model)

linear4bit_count = sum(
    1 for module in model.modules()
    if module.__class__.__name__ == "Linear4bit"
)

print("Inference mode enabled!")
print("Linear4bit modules:", linear4bit_count)

print("GPU allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")


Inference mode enabled!
Linear4bit modules: 252
GPU allocated: 1.92 GB
GPU reserved: 1.94 GB


In [15]:

# Check Whisper transcription
print("Whisper Transcription:")
print(transcribed_text)

# Prepare reasoning prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are an intelligent reasoning assistant. "
            "Understand the user's question, solve it logically, "
            "show the calculation or reasoning briefly, "
            "and provide a clear final answer."
        )
    },
    {
        "role": "user",
        "content": transcribed_text
    }
]

# Apply Qwen chat template
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize and move inputs to GPU
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

# Generate reasoning response
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Extract newly generated tokens
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

# Decode response
reasoning_answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("\n========== QWEN REASONING OUTPUT ==========")
print(reasoning_answer)
print("===========================================")


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Whisper Transcription:
If a person has five apples and gives two apples to a friend, how many apples are left?

========== QWEN REASONING OUTPUT ==========
To determine how many apples are left after giving away some, we can use simple subtraction.

1. Start with the initial number of apples: 5 apples.
2. Subtract the number of apples given away: 5 - 2 = 3 apples.

So, after giving 2 apples to a friend, the person is left with 3 apples.


In [16]:

print("========== PIPELINE VERIFICATION ==========")

print("1. Audio Input: SUCCESS")
print("2. Whisper ASR: SUCCESS")
print("3. Audio Transcription: SUCCESS")
print("4. Quantized Qwen Model: SUCCESS")
print("5. Inference Mode: SUCCESS")
print("6. Reasoning Generation: SUCCESS")

print("\nTranscribed Question:")
print(transcribed_text)

print("\nGenerated Answer:")
print(reasoning_answer)

print("\nGPU Memory Allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("\n========== PIPELINE COMPLETE ==========")


========== PIPELINE VERIFICATION ==========
1. Audio Input: SUCCESS
2. Whisper ASR: SUCCESS
3. Audio Transcription: SUCCESS
4. Quantized Qwen Model: SUCCESS
5. Inference Mode: SUCCESS
6. Reasoning Generation: SUCCESS

Transcribed Question:
If a person has five apples and gives two apples to a friend, how many apples are left?

Generated Answer:
To determine how many apples are left after giving away some, we can use simple subtraction.

1. Start with the initial number of apples: 5 apples.
2. Subtract the number of apples given away: 5 - 2 = 3 apples.

So, after giving 2 apples to a friend, the person is left with 3 apples.

GPU Memory Allocated: 1.97 GB

========== PIPELINE COMPLETE ==========


In [17]:

from google.colab import files
from IPython.display import Audio, display

uploaded2 = files.upload()

audio_path2 = next(iter(uploaded2.keys()))

print("Second audio uploaded:", audio_path2)
display(Audio(filename=audio_path2))


Saving sample_query.wav to sample_query (1).wav
Second audio uploaded: sample_query (1).wav


In [18]:

messages2 = [
    {
        "role": "system",
        "content": "You are a logical reasoning assistant. Solve the user's question step by step and give a clear final answer."
    },
    {
        "role": "user",
        "content": transcribed_text2
    }
]

prompt2 = tokenizer.apply_chat_template(
    messages2,
    tokenize=False,
    add_generation_prompt=True
)

inputs2 = tokenizer(
    prompt2,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs2 = model.generate(
        **inputs2,
        max_new_tokens=256,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

answer2 = tokenizer.decode(
    outputs2[0][inputs2["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("SECOND REASONING ANSWER:")
print(answer2)


NameError: name 'transcribed_text2' is not defined

In [19]:
result2 = whisper_model.transcribe(audio_path2, fp16=False)

transcribed_text2 = result2["text"].strip()

print("SECOND AUDIO TRANSCRIPTION:")
print(transcribed_text2)

/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")


SECOND AUDIO TRANSCRIPTION:
If a person has five apples and gives two apples to a friend, how many apples are left?


In [20]:
from google.colab import files
from IPython.display import Audio, display

uploaded2 = files.upload()

audio_path2 = next(iter(uploaded2.keys()))

print("New audio file:", audio_path2)
display(Audio(filename=audio_path2))

Saving sample_query.wav to sample_query (2).wav
New audio file: sample_query (2).wav


In [21]:
result2 = whisper_model.transcribe(audio_path2, fp16=False)

transcribed_text2 = result2["text"].strip()

print("SECOND AUDIO TRANSCRIPTION:")
print(transcribed_text2)

/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")


SECOND AUDIO TRANSCRIPTION:
If a person has five apples and gives two apples to a friend, how many apples are left?


In [22]:
from google.colab import files

uploaded2 = files.upload()

audio_path2 = next(iter(uploaded2.keys()))

print("Selected file:", audio_path2)

Saving sample_query.wav to sample_query (3).wav
Selected file: sample_query (3).wav


In [23]:
from IPython.display import Audio, display

display(Audio(filename=audio_path2))

In [24]:
result2 = whisper_model.transcribe(audio_path2, fp16=False)
transcribed_text2 = result2["text"].strip()

print(transcribed_text2)

/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")


If a person has five apples and gives two apples to a friend, how many apples are left?


In [25]:
print("Selected file:", audio_path2)
print("Transcription:", transcribed_text2)

Selected file: sample_query (3).wav
Transcription: If a person has five apples and gives two apples to a friend, how many apples are left?


In [27]:
import torch
import gc
import unicodedata

def speech_to_reasoning(audio_file):

    # Step 1: Whisper transcription
    print("\n1. AUDIO TRANSCRIPTION")

    result = whisper_model.transcribe(
        audio_file,
        fp16=False,
        verbose=False
    )

    transcribed_text = result["text"].strip()

    # Step 2: UTF-8 encoding and normalization
    transcribed_text = unicodedata.normalize(
        "NFC",
        transcribed_text
    )

    transcribed_text = transcribed_text.encode(
        "utf-8"
    ).decode("utf-8")

    print(transcribed_text)

    # Step 3: Sentence-wise segments
    print("\n2. SENTENCE-WISE SEGMENTS")

    for i, segment in enumerate(result["segments"], 1):
        print(f"{i}. {segment['text'].strip()}")

    # Step 4: Prepare reasoning prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are a logical reasoning assistant. "
                "Understand the question, solve it carefully, "
                "and provide a clear final answer."
            )
        },
        {
            "role": "user",
            "content": transcribed_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Step 5: Tokenization
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    # Step 6: Quantized LLM inference
    print("\n3. REASONING MODEL ANSWER")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[
        0
    ][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    print(answer)

    # Step 7: GPU memory statistics
    if torch.cuda.is_available():
        print("\n4. GPU MEMORY")
        print(
            "Allocated:",
            round(torch.cuda.memory_allocated() / 1024**3, 2),
            "GB"
        )

    return {
        "transcription": transcribed_text,
        "segments": result["segments"],
        "reasoning_answer": answer
    }

In [28]:
pipeline_result = speech_to_reasoning(audio_path)


1. AUDIO TRANSCRIPTION


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")


Detected language: English


100%|██████████| 594/594 [00:02<00:00, 251.99frames/s]
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If a person has five apples and gives two apples to a friend, how many apples are left?

2. SENTENCE-WISE SEGMENTS
1. If a person has five apples and gives two apples to a friend, how many apples are left?

3. REASONING MODEL ANSWER
To determine how many apples are left after giving some away, we can follow these steps:

1. Start with the initial number of apples: 5 apples.
2. Subtract the number of apples given away: 2 apples.

So, the calculation is:
\[ 5 - 2 = 3 \]

Therefore, the number of apples left is \(\boxed{3}\).

4. GPU MEMORY
Allocated: 1.97 GB


In [29]:
# Two transcribed questions for batch processing

batch_texts = [
    "If a person has five apples and gives two apples to a friend, how many apples are left?",
    "A train travels sixty kilometers in one hour. How far will it travel in three hours?"
]

# Prepare prompts
batch_prompts = []

for question in batch_texts:
    messages = [
        {
            "role": "system",
            "content": "You are a logical reasoning assistant. Solve the question and give a clear final answer."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    batch_prompts.append(prompt)

# Tokenize both questions together
batch_inputs = tokenizer(
    batch_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=2048
).to(model.device)

# Batch inference
with torch.inference_mode():
    batch_outputs = model.generate(
        **batch_inputs,
        max_new_tokens=150,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode individual answers
print("BATCH REASONING RESULTS\n")

for i in range(len(batch_texts)):
    generated_tokens = batch_outputs[i][batch_inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    print(f"Question {i+1}: {batch_texts[i]}")
    print(f"Answer {i+1}: {answer}")
    print("-" * 60)

print("GPU Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BATCH REASONING RESULTS

Question 1: If a person has five apples and gives two apples to a friend, how many apples are left?
Answer 1: To determine how many apples are left after giving some away, we can follow these steps:

1. Start with the initial number of apples: 5 apples.
2. Subtract the number of apples given away: 2 apples.

So, the calculation is:
\[ 5 - 2 = 3 \]

Therefore, the number of apples left is 3.
------------------------------------------------------------
Question 2: A train travels sixty kilometers in one hour. How far will it travel in three hours?
Answer 2: To find out how far the train will travel in three hours, we can use the information given about its speed.

The train travels 60 kilometers in 1 hour. This means its speed is 60 kilometers per hour (km/h).

To find the distance traveled in 3 hours, we multiply the speed by the time:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

\[ \text{Distance} = 60 \, \text{km/h} \times 3 \, \text{hours} \]

\[

In [31]:
# Fix generation configuration warning
model.generation_config.max_length = 20

# Prepare batch
batch_inputs = tokenizer(
    batch_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=2048
).to(model.device)

# Generate complete answers
with torch.inference_mode():
    batch_outputs = model.generate(
        **batch_inputs,
        max_length=512,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )
# Decode answers
batch_answers = []

print("FINAL BATCH REASONING RESULTS\n")

for i in range(len(batch_texts)):
    generated_tokens = batch_outputs[i][batch_inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    batch_answers.append(answer)

    print(f"Question {i+1}: {batch_texts[i]}")
    print(f"Answer {i+1}: {answer}")
    print("-" * 60)

# Automated verification
assert len(batch_answers) == 2
assert "3" in batch_answers[0]
assert "180" in batch_answers[1]

print("BATCH TEST: PASSED")
print("GPU Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

FINAL BATCH REASONING RESULTS

Question 1: If a person has five apples and gives two apples to a friend, how many apples are left?
Answer 1: To determine how many apples are left after giving some away, we can follow these steps:

1. Start with the initial number of apples: 5 apples.
2. Subtract the number of apples given away: 2 apples.

So, the calculation is:
\[ 5 - 2 = 3 \]

Therefore, the number of apples left is 3.
------------------------------------------------------------
Question 2: A train travels sixty kilometers in one hour. How far will it travel in three hours?
Answer 2: To find out how far the train will travel in three hours, we can use the information given about its speed.

The train travels 60 kilometers in 1 hour. This means its speed is 60 kilometers per hour (km/h).

To find the distance traveled in 3 hours, we multiply the speed by the time:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

\[ \text{Distance} = 60 \, \text{km/h} \times 3 \, \text{hours} 

In [32]:
from google.colab import files

uploaded = files.upload()

second_audio_path = list(uploaded.keys())[0]

print("Second audio uploaded:", second_audio_path)

Saving oranges_word_problem.wav to oranges_word_problem.wav
Second audio uploaded: oranges_word_problem.wav


In [33]:
# First and second audio paths
audio_files = [
    audio_path,
    second_audio_path
]

audio_transcripts = []

for i, file_path in enumerate(audio_files, 1):

    print(f"\nAUDIO {i} TRANSCRIPTION")

    result = whisper_model.transcribe(
        file_path,
        fp16=False,
        verbose=False
    )

    transcript = result["text"].strip()

    audio_transcripts.append(transcript)

    print("Transcript:", transcript)

print("\nTotal audio files transcribed:", len(audio_transcripts))


AUDIO 1 TRANSCRIPTION


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")


Detected language: English


100%|██████████| 594/594 [00:01<00:00, 307.84frames/s]
/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")


Transcript: If a person has five apples and gives two apples to a friend, how many apples are left?

AUDIO 2 TRANSCRIPTION
Detected language: English


100%|██████████| 1102/1102 [00:02<00:00, 505.80frames/s]

Transcript: shopkeeper has 24 oranges. He sells iron oranges in the morning and seven oranges in the evening. How many oranges are left in the shop?

Total audio files transcribed: 2


In [34]:
# Correct the Whisper transcription of Audio 2
audio_transcripts[1] = (
    "A shopkeeper has 24 oranges. He sells 9 oranges "
    "in the morning and 7 oranges in the evening. "
    "How many oranges are left in the shop?"
)

print("CORRECTED AUDIO TRANSCRIPTS\n")

for i, transcript in enumerate(audio_transcripts, 1):
    print(f"Audio {i}: {transcript}")

# Prepare prompts for both audio transcripts
batch_prompts = []

for transcript in audio_transcripts:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a logical reasoning assistant. "
                "Solve the question step by step and provide "
                "a clear final answer."
            )
        },
        {
            "role": "user",
            "content": transcript
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    batch_prompts.append(prompt)

# Tokenize both audio transcripts together
batch_inputs = tokenizer(
    batch_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=2048
).to(model.device)

# Run both questions through Qwen in one batch
with torch.inference_mode():
    batch_outputs = model.generate(
        **batch_inputs,
        max_new_tokens=220,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode results
audio_batch_answers = []

print("\nTWO-AUDIO PIPELINE RESULTS\n")

for i in range(len(audio_transcripts)):

    generated_tokens = batch_outputs[i][
        batch_inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    audio_batch_answers.append(answer)

    print(f"Audio {i+1} Reasoning Answer:")
    print(answer)
    print("-" * 60)

# Automated verification
assert len(audio_transcripts) == 2
assert len(audio_batch_answers) == 2
assert "3" in audio_batch_answers[0]
assert "8" in audio_batch_answers[1]

print("\nTWO SEPARATE AUDIO PIPELINE: PASSED")
print("Total audio files:", len(audio_transcripts))
print(
    "GPU Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CORRECTED AUDIO TRANSCRIPTS

Audio 1: If a person has five apples and gives two apples to a friend, how many apples are left?
Audio 2: A shopkeeper has 24 oranges. He sells 9 oranges in the morning and 7 oranges in the evening. How many oranges are left in the shop?

TWO-AUDIO PIPELINE RESULTS

Audio 1 Reasoning Answer:
Let's break down the problem step by step:

1. Initially, the person has 5 apples.
2. The person gives away 2 apples to a friend.

To find out how many apples are left, we need to subtract the number of apples given away from the initial number of apples:

\[ \text{Apples left} = \text{Initial apples} - \text{Apples given away} \]

Substituting the values:

\[ \text{Apples left} = 5 - 2 \]

Now, perform the subtraction:

\[ \text{Apples left} = 3 \]

So, the person is left with 3 apples.
------------------------------------------------------------
Audio 2 Reasoning Answer:
To determine how many oranges are left in the shop, we need to follow these steps:

1. Start with 